# Real-World Dataset Analysis Project — Diabetes Progression

## Project Overview
This project analyzes the **Diabetes dataset** distributed through scikit-learn. It is a real-world regression dataset originally used to study disease progression one year after baseline.

The objective is to inspect the dataset, clean and validate it, perform exploratory data analysis (EDA), quantify relationships between variables, visualize important patterns, and convert the results into practical analytical observations.

> **Important:** The target represents a quantitative measure of disease progression. This project is an educational data-analysis exercise and is **not a medical diagnostic model**.

### Dataset source and context
The dataset is available through `sklearn.datasets.load_diabetes`, which packages the well-known diabetes dataset for machine-learning experimentation. The observations contain ten baseline variables: age, sex, body mass index (BMI), average blood pressure, and six blood-serum measurements. The target is a quantitative measure of disease progression one year after baseline.

### Main questions
1. What is the structure and quality of the dataset?
2. Are there missing values or duplicate observations?
3. Which variables have the strongest linear relationship with the target?
4. How are the numerical variables distributed?
5. Do higher BMI groups show different average target values?
6. What potential outliers or unusual observations should be investigated?

## 1. Environment Setup

Install the dependencies from `requirements.txt` and run the notebook with Jupyter:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate

pip install -r requirements.txt
jupyter notebook
```

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Load the Real-World Dataset

The dataset is loaded from scikit-learn. The ten predictor variables are standardized numerical measurements. This means their values are centered/scaled rather than expressed in their original physical units.

The target is a continuous numerical value representing disease progression.

In [ ]:
dataset = load_diabetes(as_frame=True)
df = dataset.frame.copy()
df.columns = [c.strip().lower() for c in df.columns]

print("Dataset shape:", df.shape)
display(df.head())

## 3. Dataset Structure and Data Types

First, inspect columns, data types, summary statistics, and uniqueness. Understanding structure before analysis helps prevent incorrect assumptions and inappropriate transformations.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nShape:", df.shape)

print("\nDescriptive statistics:")
display(df.describe().T)

## 4. Data Quality Assessment

The analysis checks:
- Missing values
- Duplicate rows
- Invalid numeric values
- Potentially suspicious observations

Because this dataset is already curated for machine-learning use, major structural problems are not expected. Nevertheless, explicitly checking them is an important reproducible workflow step.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()

print("Missing values:")
display(missing.to_frame("missing_count"))

print("Duplicate rows:", duplicate_count)

numeric_columns = df.select_dtypes(include=np.number).columns
invalid_counts = (df[numeric_columns].isna()).sum()

print("Numeric columns:", len(numeric_columns))
print("Rows with any missing value:", int(df.isna().any(axis=1).sum()))

## 5. Cleaning and Preprocessing

### Handling missing values
No missing values are expected in the packaged dataset. If missing values were present, the strategy would depend on their amount and meaning. For example, numerical variables could use a documented median imputation when appropriate.

### Handling duplicates
Duplicate rows are checked and removed only if confirmed to be accidental duplicates. We do not blindly drop records because repeated observations can sometimes be meaningful.

### Data types
All analytical variables are numerical and already stored in numeric formats.

### Outliers
Potential outliers are identified with the IQR rule. An IQR flag is **not automatically a reason to delete a row**. In a real-world project, unusual values should first be investigated for data-entry errors, measurement issues, or legitimate extreme cases.

In [ ]:
clean_df = df.copy()

# Demonstration of safe cleaning logic
if clean_df.isna().any().any():
    for col in clean_df.select_dtypes(include=np.number).columns:
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
after = len(clean_df)

print("Rows before duplicate handling:", before)
print("Rows after duplicate handling:", after)
print("Missing values after cleaning:", int(clean_df.isna().sum().sum()))

## 6. Outlier Detection with the IQR Method

The following code counts observations outside 1.5 × IQR below Q1 or above Q3 for each numerical variable. These are screening flags rather than automatic deletion rules.

In [ ]:
outlier_summary = []

for col in clean_df.columns:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((clean_df[col] < lower) | (clean_df[col] > upper)).sum())
    outlier_summary.append([col, q1, q3, lower, upper, count])

outlier_summary = pd.DataFrame(
    outlier_summary,
    columns=["feature", "q1", "q3", "lower_bound", "upper_bound", "outlier_count"]
).sort_values("outlier_count", ascending=False)

display(outlier_summary)

## 7. Exploratory Data Analysis — Distributions

Histograms help understand the spread, concentration, and possible skewness of the variables. Because the predictors are standardized, their numeric scale is not directly comparable to the target's scale.

In [ ]:
plot_cols = list(clean_df.columns)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].hist(clean_df[col], bins=25)
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

for j in range(len(plot_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 8. Box Plots for Spread and Potential Outliers

Box plots provide a compact view of the median, interquartile range, and potential extreme observations.

In [ ]:
plt.figure(figsize=(14, 6))
clean_df.boxplot(rot=45)
plt.title("Box Plots of Dataset Variables")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

## 9. Relationship Between Important Features and Target

Scatter plots are used to inspect linear or nonlinear patterns between selected predictors and disease progression.

In [ ]:
selected = ["bmi", "bp", "s5", "age"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.ravel(), selected):
    ax.scatter(clean_df[col], clean_df["target"], alpha=0.65)
    ax.set_title(f"{col.upper()} vs Target")
    ax.set_xlabel(col)
    ax.set_ylabel("Target")

plt.tight_layout()
plt.show()

## 10. Correlation Analysis

Pearson correlation measures the strength and direction of a **linear association**. It does not prove causation.

The correlation matrix is used to identify variables that may deserve deeper investigation.

In [ ]:
corr_matrix = clean_df.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

target_correlations = corr_matrix["target"].drop("target").sort_values(
    key=lambda s: s.abs(), ascending=False
)
display(target_correlations.to_frame("correlation_with_target"))

## 11. BMI Quartile Analysis

To create an easy-to-communicate group comparison, BMI is divided into four approximately equal-sized groups. We then compare the mean target value across those groups.

This is descriptive analysis only; it should not be interpreted as proof that BMI causes disease progression.

In [ ]:
clean_df["bmi_quartile"] = pd.qcut(clean_df["bmi"], q=4, duplicates="drop")

bmi_summary = (
    clean_df.groupby("bmi_quartile", observed=True)["target"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

display(bmi_summary)

plt.figure(figsize=(10, 6))
plt.bar(bmi_summary["bmi_quartile"].astype(str), bmi_summary["mean"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Mean target")
plt.xlabel("BMI quartile")
plt.title("Average Target by BMI Quartile")
plt.tight_layout()
plt.show()

## 12. Additional Statistical Summary

The following table provides selected descriptive statistics for the main analytical variables.

In [ ]:
summary = clean_df[["age", "bmi", "bp", "s5", "target"]].agg(
    ["mean", "median", "std", "min", "max"]
).T

display(summary)

## 13. Key Findings and Interpretation

Based on the exploratory analysis:

- The dataset contains **442 observations and 11 columns**: ten predictors plus the target.
- The packaged dataset contains no missing values and no duplicate rows after quality checks.
- Variables are numerical and already provided in a standardized form.
- The strongest linear relationships with the target are concentrated among a small number of predictors rather than being evenly distributed across all features.
- BMI shows a positive association with the target, and the BMI-quartile comparison provides an intuitive group-level view of that pattern.
- Box plots and IQR screening identify potential extreme observations. These should be investigated rather than automatically removed.
- Correlation indicates association, not causation. Domain knowledge and further statistical modeling would be required before making causal or medical claims.

### Practical analytical recommendations
1. Use the strongest candidate features as inputs for a baseline regression model.
2. Compare linear regression with regularized models such as Ridge or Lasso.
3. Validate models with train/test splits or cross-validation.
4. Investigate flagged outliers before deciding whether to transform or exclude them.
5. Preserve the data dictionary and preprocessing steps so the analysis can be reproduced.
6. Avoid using this educational dataset analysis as a clinical decision tool.

## 14. Conclusion

This project demonstrates a complete real-world data-analysis workflow: dataset selection and context, structural inspection, data-quality checks, preprocessing, outlier screening, exploratory analysis, statistical summaries, visualization, correlation analysis, interpretation, and recommendations.

The most important lesson is that good data science is not only about creating charts. A reliable analysis requires understanding the dataset, documenting assumptions, checking data quality, choosing appropriate methods, and communicating limitations clearly.

## 15. Reproducibility Checklist

- [x] Real-world dataset from a reliable, documented source
- [x] Dataset context and analytical objective
- [x] Structure and data-type inspection
- [x] Missing-value assessment
- [x] Duplicate-record assessment
- [x] Data cleaning and preprocessing
- [x] IQR-based outlier screening
- [x] Pandas and NumPy analysis
- [x] Statistical summaries
- [x] Histograms
- [x] Box plots
- [x] Scatter plots
- [x] Bar chart
- [x] Correlation analysis
- [x] Findings and recommendations
- [x] Limitations and responsible interpretation